In [1]:
import pandas as pd
# import requests
from bs4 import BeautifulSoup
import time
import random
import datetime
import re
from curl_cffi import requests 

In [2]:
# 1. Siapkan headers lengkap untuk menyamar sebagai browser sungguhan
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://www.google.com/",
    "Connection": "keep-alive"
}

tanggal_hari_ini = datetime.date.today().strftime("%Y-%m-%d")

def buka_pintu():
    # 2. Buat session agar cookies (PHPSESSID) tersimpan otomatis
    session = requests.Session(impersonate="chrome")
    session.headers.update(headers)
    print("sedang membuka pintu")
    session.get('https://21cineplex.com/')
    print('pintu berhasil dibuka')
    time.sleep(2)
    return session



In [3]:
sesi = buka_pintu()


sedang membuka pintu
pintu berhasil dibuka


In [4]:
def scrape_city(session):
    response = session.get('https://21cineplex.com/theaters')
    soup  = BeautifulSoup(response.text, "html.parser")

    semua_kota = soup.find("select", id="cityChanged")
    
    detail_kota = []

    for kota in semua_kota.find_all("option"):
        id_kota = kota["data-id"]
        nama_kota = kota.text
        link_kota = kota["value"]
        detail_kota.append({
            "id_city" : id_kota,
            "city" : nama_kota,
            "link" : link_kota
        })

    print(f"semua kota berhasil discraping. total : {len(detail_kota)} Kota")
    return detail_kota 

In [5]:
daftar_kota = scrape_city(sesi)

semua kota berhasil discraping. total : 71 Kota


In [6]:
def scrape_theater(session, detail_kota):
    theater_per_kota = []

    for kota in detail_kota:
        nama_kota = kota['city']
        link_kota = kota['link']
        # sesi_kota = requests.Session()
        # sesi_kota.headers.update(headers)
        response = session.get(link_kota)
        print(f"mengambil data dari url : {link_kota}")

        soup = BeautifulSoup(response.text,"html.parser")

        t_bioskop = soup.find("tbody", id="tableXXI")

        for bioskop in t_bioskop.find_all("td",class_="tg-c3ow"):
            nama = bioskop.text
            link_theater = bioskop.find("a")["href"]
            id_bioskop = link_theater.split(",")[1]
            theater_per_kota.append({
                "id_bioskop" : id_bioskop,
                "nama bioskop" : nama,
                "kota" : nama_kota,
                "link_bioskop" : link_theater})
        
        #memberi jeda acak antara 3 detik hingga 6.5 detik
        jeda = random.uniform(3.0 , 6.5)
        print(f'Istirahat sejenak {jeda:.2f} detik...')
        time.sleep(jeda)

    return theater_per_kota

In [7]:
daftar_bioskop = scrape_theater(sesi, daftar_kota)

mengambil data dari url : https://21cineplex.com/theaters/daftar-bioskop-21-di-ambon,77.htm
Istirahat sejenak 4.48 detik...
mengambil data dari url : https://21cineplex.com/theaters/daftar-bioskop-21-di-balikpapan,79.htm
Istirahat sejenak 5.00 detik...
mengambil data dari url : https://21cineplex.com/theaters/daftar-bioskop-21-di-bandung,12.htm
Istirahat sejenak 5.41 detik...
mengambil data dari url : https://21cineplex.com/theaters/daftar-bioskop-21-di-banjarmasin,76.htm
Istirahat sejenak 5.93 detik...
mengambil data dari url : https://21cineplex.com/theaters/daftar-bioskop-21-di-batam,27.htm
Istirahat sejenak 4.73 detik...
mengambil data dari url : https://21cineplex.com/theaters/daftar-bioskop-21-di-bekasi,36.htm
Istirahat sejenak 5.50 detik...
mengambil data dari url : https://21cineplex.com/theaters/daftar-bioskop-21-di-bengkulu,81.htm
Istirahat sejenak 6.05 detik...
mengambil data dari url : https://21cineplex.com/theaters/daftar-bioskop-21-di-bogor,26.htm
Istirahat sejenak 5.55 

In [8]:
daftar_bioskop

[{'id_bioskop': '398',
  'nama bioskop': 'AMBON CITY CENTER XXI',
  'kota': 'AMBON',
  'link_bioskop': 'https://21cineplex.com/theater/bioskop-ambon-city-center-xxi,398,ABNAMCC.htm'},
 {'id_bioskop': '297',
  'nama bioskop': 'E-WALK XXI',
  'kota': 'BALIKPAPAN',
  'link_bioskop': 'https://21cineplex.com/theater/bioskop-ewalk-xxi,297,BLPEWAL.htm'},
 {'id_bioskop': '449',
  'nama bioskop': 'PENTACITY XXI',
  'kota': 'BALIKPAPAN',
  'link_bioskop': 'https://21cineplex.com/theater/bioskop-pentacity-xxi,449,BLPPECI.htm'},
 {'id_bioskop': '276',
  'nama bioskop': 'STUDIO XXI',
  'kota': 'BALIKPAPAN',
  'link_bioskop': 'https://21cineplex.com/theater/bioskop-studio-xxi,276,BLPSTUD.htm'},
 {'id_bioskop': '231',
  'nama bioskop': 'BRAGA XXI',
  'kota': 'BANDUNG',
  'link_bioskop': 'https://21cineplex.com/theater/bioskop-braga-xxi,231,BDGBRAG.htm'},
 {'id_bioskop': '249',
  'nama bioskop': 'CIWALK XXI',
  'kota': 'BANDUNG',
  'link_bioskop': 'https://21cineplex.com/theater/bioskop-ciwalk-xxi,249

In [9]:
def scrape_jadwal(sesi_scraping, daftar_bioskop):
    hasil_semua_jadwal = []
    
    for bioskop in daftar_bioskop:
        nama_bioskop = bioskop['nama bioskop']
        kota = bioskop['kota']
        id_bioskop = bioskop['id_bioskop']
        link = bioskop['link_bioskop']
        
        print(f"Sedang scraping: {nama_bioskop} ({kota})...")
        
        try:
            # 1. Request halaman bioskop
            response = sesi_scraping.get(link)
            soup = BeautifulSoup(response.text, "html.parser")
            
            # 2. Cari wadah daftar film
            list_movies = soup.find("div", class_="list-movies")
            
            # PENGAMAN 1: Pastikan wadah film ada
            if list_movies:
                detail_per_movie = list_movies.find_all("div", class_="list-movie")
                
                for movie in detail_per_movie:
                    # --- A. Ambil Judul ---
                    judul_tag = movie.find("h3")
                    judul = judul_tag.text.strip() if judul_tag else "Judul Tidak Diketahui"
                    
                    # --- B. Ambil Durasi & Rating ---
                    poster_div = movie.find("div", class_="list-movie-poster")
                    durasi = "N/A"
                    rating_usia = "N/A"
                    
                    if poster_div:
                        semua_p = poster_div.find_all("p")
                        if len(semua_p) > 0:
                            durasi = semua_p[0].text.strip()
                            
                        if len(semua_p) > 1:
                            img_rating = semua_p[1].find("img")
                            if img_rating:
                                rating_usia = img_rating.get("alt")
                                
                    # --- C. Ambil Jam Tayang ---
                    ul_jam = movie.find("ul", class_="time-cinema")
                    
                    if ul_jam:
                        for li in ul_jam.find_all("li"):
                            tag_a = li.find("a")
                            if tag_a:
                                jam = tag_a.text.strip()
                                
                                # Cek status active/closed
                                classes = li.get("class", [])
                                status = "active" if "active" in classes else "closed"
                                
                                # Masukkan ke list utama
                                hasil_semua_jadwal.append({
                                    "id_bioskop": id_bioskop,
                                    "nama_bioskop": nama_bioskop,
                                    "kota": kota,
                                    "judul_film": judul,
                                    "durasi": durasi,
                                    "rating_usia": rating_usia,
                                    "jam_tayang": jam,
                                    "status_tiket": status
                                })
                    else:
                        # Kalau filmnya ada tapi belum ada jam tayang (Coming Soon)
                        hasil_semua_jadwal.append({
                            "id_bioskop": id_bioskop,
                            "nama_bioskop": nama_bioskop,
                            "kota": kota,
                            "judul_film": judul,
                            "durasi": durasi,
                            "rating_usia": rating_usia,
                            "jam_tayang": "N/A",
                            "status_tiket": "N/A"
                        })
            else:
                # Kalau bioskop ini nggak punya tabel list-movies
                print(f"  [Skip] Tidak ada jadwal film di {nama_bioskop}")
                
        except Exception as e:
            # PENGAMAN 2: Kalau request gagal/timeout, lanjut ke bioskop berikutnya
            print(f"  [Error] Gagal scraping {nama_bioskop}: {e}")
            
        # Jeda acak antar bioskop
        jeda = random.uniform(3.0, 6.5)
        time.sleep(jeda)
        
    return hasil_semua_jadwal

In [10]:
daftar_schedule = scrape_jadwal(sesi,daftar_bioskop)

Sedang scraping: AMBON CITY CENTER XXI (AMBON)...
Sedang scraping: E-WALK XXI (BALIKPAPAN)...
Sedang scraping: PENTACITY XXI (BALIKPAPAN)...
Sedang scraping: STUDIO XXI (BALIKPAPAN)...
Sedang scraping: BRAGA XXI (BANDUNG)...
Sedang scraping: CIWALK XXI (BANDUNG)...
Sedang scraping: d’botanica XXI (BANDUNG)...
Sedang scraping: EMPIRE XXI (BANDUNG)...
Sedang scraping: FESTIVAL CITYLINK XXI (BANDUNG)...
Sedang scraping: JATOS XXI (BANDUNG)...
Sedang scraping: SUMMARECON MALL BANDUNG XXI (BANDUNG)...
Sedang scraping: TENTH AVENUE XXI (BANDUNG)...
Sedang scraping: THEE MATIC MALL XXI (BANDUNG)...
Sedang scraping: TRANSMART BUAH BATU XXI (BANDUNG)...
Sedang scraping: TSM XXI (BANDUNG)...
Sedang scraping: UBERTOS XXI (BANDUNG)...
Sedang scraping: STUDIO XXI (BANJARMASIN)...
Sedang scraping: BCS XXI (BATAM)...
Sedang scraping: MEGA XXI (BATAM)...
Sedang scraping: NAGOYA HILL MALL XXI (BATAM)...
Sedang scraping: ONE BATAM MALL XXI (BATAM)...
Sedang scraping: PANBIL XXI (BATAM)...
Sedang scrapin

In [11]:
daftar_schedule

[{'id_bioskop': '398',
  'nama_bioskop': 'AMBON CITY CENTER XXI',
  'kota': 'AMBON',
  'judul_film': 'Pulang Kampung',
  'durasi': '114 Minutes',
  'rating_usia': 'SEMUA UMUR',
  'jam_tayang': '12:00',
  'status_tiket': 'active'},
 {'id_bioskop': '398',
  'nama_bioskop': 'AMBON CITY CENTER XXI',
  'kota': 'AMBON',
  'judul_film': 'Pulang Kampung',
  'durasi': '114 Minutes',
  'rating_usia': 'SEMUA UMUR',
  'jam_tayang': '14:15',
  'status_tiket': 'active'},
 {'id_bioskop': '398',
  'nama_bioskop': 'AMBON CITY CENTER XXI',
  'kota': 'AMBON',
  'judul_film': 'Pulang Kampung',
  'durasi': '114 Minutes',
  'rating_usia': 'SEMUA UMUR',
  'jam_tayang': '16:30',
  'status_tiket': 'active'},
 {'id_bioskop': '398',
  'nama_bioskop': 'AMBON CITY CENTER XXI',
  'kota': 'AMBON',
  'judul_film': 'Pulang Kampung',
  'durasi': '114 Minutes',
  'rating_usia': 'SEMUA UMUR',
  'jam_tayang': '18:45',
  'status_tiket': 'active'},
 {'id_bioskop': '398',
  'nama_bioskop': 'AMBON CITY CENTER XXI',
  'kota': 

In [12]:
def scrape_now_playing_per_kota(session):
        response = session.get('https://21cineplex.com')
        # 3. Ambil token (pakai home_html yang baru)
        home_html = response.text
        home_soup = BeautifulSoup(home_html,"html.parser")
        match = re.search(r'token:\s*"([^"]+)"', home_html)

        if match:
            token = match.group(1)
            print("✅ Token berhasil didapat!")
            print(f"Token: {token[:15]}...") # Print sebagian aja
        else:
            print("❌ Masih gagal dapet token. Coba cek lagi.")

        # STEP 3 - Ambil semua kota

        city_dropdown = home_soup.find("select", id="cityChanged")

        cities = []

        for city in city_dropdown.find_all("option"):

            cities.append({
                "city_id": city["value"],
                "city_name": city.text.strip()
            })

        print(f"Total kota: {len(cities)}")

        # STEP 4 - Loop semua kota
        all_movies = []

        for city in cities:

            print(f"Processing: {city['city_name']}")

            payload = {
                "data": "getMovNowplaying",
                "city_id": city["city_id"],
                "token": token
            }

            response = session.post(
                "https://21cineplex.com/script/ajax-home-nowplaying.php",
                data=payload,
                headers={
                    "Referer": "https://21cineplex.com/",
                    "X-Requested-With": "XMLHttpRequest"
                })

            try:
                ajax_data = response.json()

            except Exception:

                print(f"Gagal di {city['city_name']}")

                continue

            ajax_soup = BeautifulSoup(ajax_data["msg"], "html.parser")

            movie_cards = ajax_soup.find_all("div", class_="movie")

            for movie in movie_cards:

                try:

                    title = movie.find("h4").text.strip()

                    link = movie.find("a")["href"]
                    
                    all_movies.append({
                        "city_id": city["city_id"],
                        "city_name": city["city_name"],
                        "title": title,
                        "link": link
                    })

                except Exception:
                    pass

            time.sleep(
            random.uniform(3.0, 6.5)
            )
        print(f"Total film per kota : {len(all_movies)}")
        return all_movies

In [13]:
daftar_now_playing_per_kota = scrape_now_playing_per_kota(sesi)

✅ Token berhasil didapat!
Token: 6915bad0aa6a309...
Total kota: 71
Processing: AMBON
Processing: BALIKPAPAN
Processing: BANDUNG
Processing: BANJARMASIN
Processing: BATAM
Processing: BEKASI
Processing: BENGKULU
Processing: BOGOR
Processing: BONDOWOSO
Processing: BONTANG
Processing: CIANJUR
Processing: CIKARANG
Processing: CILEGON
Processing: CIREBON
Processing: DENPASAR
Processing: DEPOK
Processing: DUMAI
Processing: GARUT
Processing: GORONTALO
Processing: GRESIK
Processing: INDRAMAYU
Processing: JAKARTA
Processing: JAMBI
Processing: JAYAPURA
Processing: JEMBER
Processing: KARAWANG
Processing: KEDIRI
Processing: KENDARI
Processing: KISARAN
Processing: KLATEN
Processing: KUALA KAPUAS
Processing: KUPANG
Processing: LABUAN BAJO
Processing: LAMPUNG
Processing: MADIUN
Processing: MAGELANG
Processing: MAKASSAR
Processing: MALANG
Processing: MAMUJU
Processing: MANADO
Processing: MANOKWARI
Processing: MAROS
Processing: MATARAM
Processing: MEDAN
Processing: PADANG
Processing: PALANGKARAYA
Proces

In [14]:
daftar_now_playing_per_kota

[{'city_id': '77',
  'city_name': 'AMBON',
  'title': 'The Beast',
  'link': 'https://21cineplex.com/the-beast,139048,26BEAT.htm'},
 {'city_id': '77',
  'city_name': 'AMBON',
  'title': 'Hamka: Awal Mula (Vol 3)',
  'link': 'https://21cineplex.com/hamka-awal-mula-(vol-3),139035,16HAM3.htm'},
 {'city_id': '77',
  'city_name': 'AMBON',
  'title': 'Pulang Kampung',
  'link': 'https://21cineplex.com/pulang-kampung,139027,16PKAG.htm'},
 {'city_id': '77',
  'city_name': 'AMBON',
  'title': 'People Pleaser: Si Paling Gak Enakan',
  'link': 'https://21cineplex.com/people-pleaser-si-paling-gak-enakan,138292,16PPSP.htm'},
 {'city_id': '77',
  'city_name': 'AMBON',
  'title': 'Hungry Ghost Festival',
  'link': 'https://21cineplex.com/hungry-ghost-festival,138290,16HGFL.htm'},
 {'city_id': '77',
  'city_name': 'AMBON',
  'title': 'The Hunger Games: Sunrise on the Reaping',
  'link': 'https://21cineplex.com/the-hunger-games-sunrise-on-the-reaping,138277,26HGSR.htm'},
 {'city_id': '77',
  'city_name

In [15]:
len(daftar_now_playing_per_kota)

745

In [16]:

def deduplikat_now_playing(data):
    daftar_film_now_playing_unik = []
    judul_yang_sudah_masuk = set()

    for i in daftar_now_playing_per_kota:
        judul = i['title']
        if judul not in judul_yang_sudah_masuk:
            judul_yang_sudah_masuk.add(judul)
            daftar_film_now_playing_unik.append(i)

    return daftar_film_now_playing_unik


In [17]:
daftar_film_unik = deduplikat_now_playing(daftar_now_playing_per_kota)

In [19]:
len(daftar_film_unik)

30

In [28]:
def movie_detail(data):
    detail_movie = []
    
    for i in data:
        url = i['link']
        title = i['title']
        
        try:
            response = sesi.get(url)
            soup = BeautifulSoup(response.text, "html.parser")
            desc_box = soup.find("div", class_="desc-box")
            if not desc_box:
                print(f'[Skip] Desc box tidak ketemu di film ID {title}')

            judul = desc_box.find("h2").text
            genre = desc_box.find("li",class_="movie_genre").text.replace('Jenis Film','').strip()
            desc_movie = desc_box.find_all("li",class_="movie_produser")
            produser = "NA"
            sutradara = "NA"
        
            if len(desc_movie) > 0:
                produser = desc_movie[0].text.replace("Produser","").strip()
            if len(desc_movie) > 1:
                sutradara = desc_movie[1].text.replace("Sutradara","").strip()

            penulis = desc_box.find("li",class_="movie_writer").text.replace("Penulis",'').strip()
            produksi = desc_box.find("li",class_="movie_distributor").text.replace("Produksi","").strip()

            detail_movie.append({
                'judul' : judul,
                'genre' : genre,
                'produser' : produser,
                'sutradara' : sutradara,
                'penulis' : penulis,
                'produksi' : produksi
            })

            print(f'Jumlah film yang masuk {len(detail_movie)}')
            time.sleep(random.uniform(2.0,4.0))

        except Exception as e:
            print(f'[ERROR] Gagal Scraping film {title} {i['link']}: {e}')

    return detail_movie
        



In [29]:
daftar_movie_detail = movie_detail(daftar_film_unik)

Jumlah film yang masuk 1
Jumlah film yang masuk 2
Jumlah film yang masuk 3
Jumlah film yang masuk 4
Jumlah film yang masuk 5
Jumlah film yang masuk 6
Jumlah film yang masuk 7
Jumlah film yang masuk 8
Jumlah film yang masuk 9
Jumlah film yang masuk 10
Jumlah film yang masuk 11
Jumlah film yang masuk 12
Jumlah film yang masuk 13
Jumlah film yang masuk 14
Jumlah film yang masuk 15
Jumlah film yang masuk 16
Jumlah film yang masuk 17
Jumlah film yang masuk 18
Jumlah film yang masuk 19
Jumlah film yang masuk 20
Jumlah film yang masuk 21
Jumlah film yang masuk 22
Jumlah film yang masuk 23
Jumlah film yang masuk 24
Jumlah film yang masuk 25
Jumlah film yang masuk 26
Jumlah film yang masuk 27
Jumlah film yang masuk 28
Jumlah film yang masuk 29
Jumlah film yang masuk 30


In [ ]:
daftar_movie_detail
    

[{'judul': 'The Beast',
  'genre': 'Action, Crime',
  'produser': 'Keith Kjarval, John Logan Pierson, Robert Van Norden',
  'sutradara': 'Renny Harlin',
  'penulis': 'Umair Aleem',
  'produksi': 'Unified Pictures'},
 {'judul': 'Hamka: Awal Mula (Vol 3)',
  'genre': 'Drama, Biography',
  'produser': 'Frederica, Chand Parwez Servia',
  'sutradara': 'Fajar Bustomi',
  'penulis': 'Alim Sudio, Cassandra Massardi',
  'produksi': 'Falcon Pictures, Starvision'},
 {'judul': 'Pulang Kampung',
  'genre': 'Drama, Musical',
  'produser': 'Ernest Prakasa, Dipa Andika',
  'sutradara': 'Bene Dion Rajagukguk',
  'penulis': 'Bene Dion Rajagukguk',
  'produksi': 'Imajinari'},
 {'judul': 'People Pleaser: Si Paling Gak Enakan',
  'genre': 'Drama, Comedy',
  'produser': 'Andi Suryanto, Marcella Daryanani',
  'sutradara': 'Bobby Prasetyo',
  'penulis': 'Evelyn Afnilia',
  'produksi': 'LYTO Pictures'},
 {'judul': 'Hungry Ghost Festival',
  'genre': 'Horror, Comedy',
  'produser': 'Raam Punjabi',
  'sutradara'